## [控制流](https://docs.langchain.com/oss/python/langgraph/graph-api-control-flow)

### 循环结构

#### 一、循环限制

In [ ]:
from typing import Literal, TypedDict

from langchain_core.runnables import RunnableConfig
from langgraph.graph import StateGraph, START
from langgraph.managed import RemainingSteps


class State(TypedDict):
    # 剩余步骤数，框架会自动维护
    remaining_steps: RemainingSteps

def loop_node(state: State, config: RunnableConfig):
    cur_step = config["metadata"]["langgraph_step"]
    print(f"loop_node 当前步数: {cur_step}, 剩余步数: {state['remaining_steps']}")

def route(state: State) -> Literal["loop_node", "__end__"]:
    # 主动退出循环，否则异常退出
    if state["remaining_steps"] < 3:
        print(f"剩余步数不足，退出循环")
        return "__end__"
    return "loop_node"


builder = StateGraph(State)
builder.add_node(loop_node)
builder.add_edge(START, "loop_node")
builder.add_conditional_edges("loop_node", route)
graph = builder.compile()
graph.invoke({}, config={"recursion_limit": 10})
